In [ ]:
from tools.search_tool import search_tool
from langchain.agents import create_agent
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import TypedDict
from langgraph.graph import START,END,StateGraph

In [29]:
class Blog_State(TypedDict):
    topic: str
    blog: str
    score: str
    feedback: str

In [50]:
class WriterSchema(BaseModel):
    blog: str = Field("An Analysis blog about the topic.")

In [30]:
class CritiqueSchema(BaseModel):
    score: int = Field("An integer score from 1 to 10 evaluating the quality, depth, and engagement of the blog",ge=0,le=10)
    feedback: str = Field("An detailed suggestions to improve the quality, depth, and engagement of the blog")


In [ ]:
def critique_agent(state: Blog_State):
    blog = state['blog']

    llm = OllamaLLM(model="qwen3:8b", temperature=1)
    prompt = f"""Critique the following blog and return JSON with this shape:
    {{"score": <int between 0 and 10>, "feedback": "<string>"}}

    Blog:
    {blog}
    """

    chain = llm | JsonOutputParser(pydantic_object=CritiqueSchema)
    response = chain.invoke(prompt)

    print(response)

    state['feedback'] = response['feedback']
    state['score'] = str(response['score'])

    return state


In [99]:
def writer_agent(state: Blog_State):
    topic = state['topic']

    template = """Analyse the root cause, impact and future of following topic in a blog. The blog should have multiple sections."""

    agent = create_agent(model="ollama:qwen3:8b",
    tools=[search_tool],
    system_prompt=template,response_format=WriterSchema)

    result = agent.invoke({"messages": [{"role": "user", "content": topic}]})
    

    return {'blog':result['messages'][-1].content}


In [106]:
graph = StateGraph(Blog_State)

In [107]:
graph.add_node("write",writer_agent)
graph.add_node("critique",critique_agent)

In [108]:
graph.add_edge(START,"write")
graph.add_edge("write","critique")
graph.add_edge("critique",END)

In [109]:
workflow = graph.compile()

In [110]:
initial_state = {"topic":"Ethanol in petrol in India"}

In [111]:
response = workflow.invoke(initial_state)

NotImplementedError: 

In [ ]:
response

"# Ethanol in Petrol in India: Root Cause, Impact, and Future  \n\n## Root Cause  \nIndia's push for ethanol blending in petrol stems from a combination of **environmental, economic, and political drivers**:  \n\n1. **Climate Goals**: Ethanol is a renewable fuel that reduces greenhouse gas emissions compared to conventional petrol. India’s net-zero commitments under the Paris Agreement necessitate a shift toward cleaner energy sources.  \n2. **Energy Security**: Reducing dependence on imported crude oil is a key priority. Ethanol, derived from crops like sugarcane, is domestically produced and lowers oil import bills.  \n3. **Subsidy Relief**: The government aims to reduce subsidies on fossil fuels by promoting cheaper, locally sourced ethanol.  \n4. **Policy Push**: Initiatives like the *National Biofuel Policy* (2018) and the *FAME India Scheme* (Fast & Furious) have mandated ethanol blending to meet targets (e.g., 20% ethanol in petrol by 2030).  \n\nHowever, the **political dimensi

__main__.Blog_State['blog']

In [17]:
agent = create_agent(model="ollama:qwen3:8b",
    tools=[search_tool],
    system_prompt=template
)

# agent_executor = AgentExecutor(
#     agent=agent, tools=[search_tool], verbose=True, handle_parsing_errors=True
# )



In [18]:
result = agent.invoke({"messages": [{"role": "user", "content": "Ethanol in petrol in India"}]})


In [19]:
result

{'messages': [HumanMessage(content='Ethanol in petrol in India', additional_kwargs={}, response_metadata={}, id='47bb0f03-51cc-4a75-a725-05abf8000ca7'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-07-08T11:31:59.635477Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25943113875, 'load_duration': 2470932417, 'prompt_eval_count': 164, 'prompt_eval_duration': 402280000, 'eval_count': 587, 'eval_duration': 23060595000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019f417f-5cba-7f63-b1d5-9bc5792d53a0-0', tool_calls=[{'name': 'search_tool', 'args': {'query': 'Root causes of ethanol in petrol in India'}, 'id': '37d7cf25-f42f-4778-b107-5462ec7acf24', 'type': 'tool_call'}, {'name': 'search_tool', 'args': {'query': 'Impact of ethanol blending in Indian petrol'}, 'id': '39b633b1-1737-406e-9bf1-63943a919e16', 'type': 'tool_call'}, {'name': 'search_tool', 'args': {'query': 'Future of etha

In [24]:
response = result['messages'][5].content


In [25]:
response

'# Ethanol in Petrol in India: Root Causes, Impact, and Future  \n\n## **Root Causes of Ethanol Blending in India**  \nIndia’s push for ethanol blending in petrol is driven by a mix of **economic, environmental, and geopolitical factors**:  \n\n1. **Reduction of Oil Imports**:  \n   - India is one of the world’s largest oil importers. Ethanol blending reduces dependence on crude oil, lowering import costs and improving energy security. The government aims to achieve **E20** (20% ethanol in petrol) by 2030, which could cut oil imports by 30–40 million tonnes annually.  \n\n2. **Climate Change Mitigation**:  \n   - Ethanol is a renewable fuel that emits less carbon dioxide compared to conventional petrol. The Indian government’s **Net Zero by 2070** target aligns with promoting biofuels like ethanol.  \n\n3. **Agricultural Surplus Utilization**:  \n   - Ethanol production uses surplus crops like sugarcane, maize, and rice, which are otherwise used for food. This aims to **reduce food pri